In [20]:
import sys
sys.path.insert(0, r"d:\py_radno\Semantra\semantra_agent\src")
print('Added semantra_agent/src to sys.path:', sys.path[0])

Added semantra_agent/src to sys.path: d:\py_radno\Semantra\semantra_agent\src


# Semantra Core SDK — Basics

This notebook demonstrates the basic usage of the `semantra-core` SDK:

- Pydantic models for schemas, mappings, and knowledge
- In-memory reference implementations of the service protocols
- No backend or external services required

In [21]:
# Install the SDK in editable mode (run once)
#!pip install -e ../semantra-core

In [22]:
from semantra_core.models.schema import (
    DatasetHandle,
    SchemaProfile,
    ColumnProfile,
)
from semantra_core.models.knowledge import CanonicalGlossaryEntry
from semantra_core.services import (
    InMemoryMappingEngine,
    InMemoryKnowledgeBase,
    BoundedLLMService,
)

## 1. Build a source dataset

In [23]:
source = DatasetHandle(
    dataset_id="customer_src",
    dataset_name="customer_src",
    schema_profile=SchemaProfile(
        dataset_id="customer_src",
        dataset_name="customer_src",
        row_count=120,
        columns=[
            ColumnProfile(
                name="cust_id",
                normalized_name="cust_id",
                dtype="str",
                null_ratio=0.0,
                unique_ratio=1.0,
                non_null_count=120,
            ),
            ColumnProfile(
                name="email",
                normalized_name="email",
                dtype="str",
                null_ratio=0.05,
                unique_ratio=0.95,
                non_null_count=114,
            ),
        ],
    ),
)
print(source)

dataset_id='customer_src' dataset_name='customer_src' schema_profile=SchemaProfile(dataset_id='customer_src', dataset_name='customer_src', row_count=120, columns=[ColumnProfile(name='cust_id', normalized_name='cust_id', description='', declared_type='', dtype='str', null_ratio=0.0, unique_ratio=1.0, avg_length=0.0, non_null_count=120, sample_values=[], distinct_sample_values=[], detected_patterns=[], tokenized_name=[]), ColumnProfile(name='email', normalized_name='email', description='', declared_type='', dtype='str', null_ratio=0.05, unique_ratio=0.95, avg_length=0.0, non_null_count=114, sample_values=[], distinct_sample_values=[], detected_patterns=[], tokenized_name=[])]) preview_rows=[]


## 2. Build a target schema

In [24]:
target = SchemaProfile(
    dataset_id="customer_tgt",
    dataset_name="customer_tgt",
    row_count=0,
    columns=[
        ColumnProfile(
            name="customer_key",
            normalized_name="customer_key",
            dtype="str",
            null_ratio=0.0,
            unique_ratio=0.0,
            non_null_count=0,
        ),
        ColumnProfile(
            name="email_address",
            normalized_name="email_address",
            dtype="str",
            null_ratio=0.0,
            unique_ratio=0.0,
            non_null_count=0,
        ),
    ],
)
print(target)

dataset_id='customer_tgt' dataset_name='customer_tgt' row_count=0 columns=[ColumnProfile(name='customer_key', normalized_name='customer_key', description='', declared_type='', dtype='str', null_ratio=0.0, unique_ratio=0.0, avg_length=0.0, non_null_count=0, sample_values=[], distinct_sample_values=[], detected_patterns=[], tokenized_name=[]), ColumnProfile(name='email_address', normalized_name='email_address', description='', declared_type='', dtype='str', null_ratio=0.0, unique_ratio=0.0, avg_length=0.0, non_null_count=0, sample_values=[], distinct_sample_values=[], detected_patterns=[], tokenized_name=[])]


## 3. Use the in-memory mapping engine

In [25]:
engine = InMemoryMappingEngine()
candidates = engine.map_source_to_target(source, target)
print(f"Got {len(candidates)} candidates (stub returns empty).")
print("Scoring signals:", engine.get_scoring_signals())

Got 1 candidates (stub returns empty).
Scoring signals: name=0.0 semantic=0.0 knowledge=0.0 canonical=0.0 pattern=0.0 statistical=0.0 overlap=0.0 embedding=0.0 correction=0.0 llm=0.0


## 4. Use the in-memory knowledge base

In [26]:
kb = InMemoryKnowledgeBase()
kb.add(
    CanonicalGlossaryEntry(
        concept_id="CUST_ID",
        entity="Customer",
        attribute="customer_id",
        display_name="Customer Identifier",
        description="Stable identifier for a customer record.",
    )
)
kb.add(
    CanonicalGlossaryEntry(
        concept_id="CUST_EMAIL",
        entity="Customer",
        attribute="email",
        display_name="Customer Email",
    )
)

print("Search 'email':", kb.search_concepts("email"))
print("Lookup CUST_ID:", kb.get_canonical_concept("CUST_ID"))

Search 'email': [CanonicalGlossaryEntry(concept_id='CUST_EMAIL', entity='Customer', attribute='email', display_name='Customer Email', description='', data_type='', aliases=[], privacy=CanonicalPrivacyMetadata(is_pii=False, is_gdpr_special_category=False, pii_categories=[], data_subject_types=[]))]
Lookup CUST_ID: concept_id='CUST_ID' entity='Customer' attribute='customer_id' display_name='Customer Identifier' description='Stable identifier for a customer record.' data_type='' aliases=[] privacy=CanonicalPrivacyMetadata(is_pii=False, is_gdpr_special_category=False, pii_categories=[], data_subject_types=[])


## 5. Use the bounded LLM service (stub)

In [27]:
llm = BoundedLLMService()
result = llm.validate_mapping(
    source_field="cust_id",
    candidate_targets=["customer_key", "customer_id", "id"],
    context={"description": "primary key"},
)
print(result)

{'selected_target': 'customer_key', 'confidence': 0.0, 'reasoning': ['LLM service not configured; using stub.']}
